# DID, COVID-inclusive, no controls (LM_AIOE + MS_SCORE)

Companion robustness check to `did_with_controls_unified.ipynb`. Two differences from that file:

1. **2020 and 2021 are kept in the panel** (the baseline analysis excludes them; this checks
   whether the main TWFE result survives their inclusion).
2. **No Job Zone control** — this is the plain no-control specification, matching the
   dissertation's baseline TWFE equation, just re-estimated on the COVID-inclusive sample.

Produces a 4-row table: 2 indices x 2 outcomes, pooled post-treatment coefficient only.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS

df = pd.read_csv('bls_onet_felten_ms_panel_harmonised.csv')
# NOTE: no COVID exclusion here on purpose -- this is the COVID-inclusive robustness sample.

df['TOT_EMP']  = pd.to_numeric(df['TOT_EMP'], errors='coerce')
df['A_MEDIAN'] = pd.to_numeric(df['A_MEDIAN'], errors='coerce')
df['log_emp']  = np.log(df['TOT_EMP'].replace(0, np.nan))
df['log_wage'] = np.log(df['A_MEDIAN'].replace(0, np.nan))
df['post']     = (df['year'] >= 2023).astype(int)

print(f'Base panel rows (COVID years included): {len(df)}')
print(f'Years present: {sorted(df["year"].unique())}')


## Config: the two indices this notebook runs through the same pipeline
Same pattern as `did_with_controls_unified.ipynb`, minus the Job Zone merge/filter.


In [ ]:
INDICES = [
    {'col': 'LM_AIOE',  'label': 'LM_AIOE'},
    {'col': 'MS_SCORE', 'label': 'MS_SCORE'},
]
OUTCOMES = ['log_emp', 'log_wage']

def prep_index_panel(base_df, idx_col):
    """Filter to rows with this index available, standardise, build did."""
    d = base_df[base_df[idx_col].notna()].copy()
    d['idx_std'] = (d[idx_col] - d[idx_col].mean()) / d[idx_col].std()
    d['did'] = d['idx_std'] * d['post']
    d['unit'] = d['OCC_CODE'] + '_' + d['NAICS'].astype(str)
    return d

panels = {ix['label']: prep_index_panel(df, ix['col']) for ix in INDICES}
for label, d in panels.items():
    print(f'{label}: {len(d)} rows, {d["OCC_CODE"].nunique()} occupations')


## Pooled DID, COVID-inclusive, no control -- both indices, both outcomes


In [ ]:
pooled_results = {}

for ix in INDICES:
    label = ix['label']
    d = panels[label].set_index(['unit', 'year'])
    for outcome in OUTCOMES:
        sample = d[[outcome, 'did']].dropna()

        model = PanelOLS.from_formula(f'{outcome} ~ did + EntityEffects + TimeEffects', data=sample)
        res = model.fit(cov_type='clustered', cluster_entity=True)

        pooled_results[(label, outcome)] = res

        print(f'=== {label} x {outcome}: COVID-inclusive, no control ===')
        print(res.summary)
        print('=' * 80)


## Event study, COVID-inclusive, no control -- both indices, both outcomes


In [ ]:
def run_es(base_df, idx_col, outcome):
    d = base_df[base_df[idx_col].notna()].copy()
    d['idx_std'] = (d[idx_col] - d[idx_col].mean()) / d[idx_col].std()
    years = [y for y in sorted(d['year'].unique()) if y != 2022]
    for y in years:
        d[f'idx_X_{y}'] = d['idx_std'] * (d['year'] == y).astype(int)
    d['unit'] = d['OCC_CODE'] + '_' + d['NAICS'].astype(str)
    d = d.set_index(['unit', 'year'])

    idx_terms = ' + '.join([f'idx_X_{y}' for y in years])
    cols = [outcome] + [f'idx_X_{y}' for y in years]
    formula = f'{outcome} ~ {idx_terms} + EntityEffects + TimeEffects'

    sample = d[cols].dropna()
    res = PanelOLS.from_formula(formula, data=sample).fit(cov_type='clustered', cluster_entity=True)
    return res, years

es_results = {}
for ix in INDICES:
    for outcome in OUTCOMES:
        res, years = run_es(df, ix['col'], outcome)
        es_results[(ix['label'], outcome)] = (res, years)
        print(f'--- Event study done: {ix["label"]} x {outcome} (COVID-inclusive) ---')


## Event study plots: one figure per index x outcome, with COVID years visible


In [ ]:
def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''

outcome_titles = {'log_emp': 'Log Employment', 'log_wage': 'Log Wage'}

for ix in INDICES:
    label = ix['label']
    for outcome in OUTCOMES:
        res, years = es_results[(label, outcome)]
        coef_names = [f'idx_X_{y}' for y in years]
        coefs = res.params[coef_names].values
        ses = res.std_errors[coef_names].values
        all_years = years + [2022]
        all_coefs = list(coefs) + [0]
        all_ses = list(ses) + [0]
        sorted_data = sorted(zip(all_years, all_coefs, all_ses))
        plot_years, plot_coefs, plot_ses = zip(*sorted_data)

        fig, ax = plt.subplots(figsize=(8, 5.5))
        ax.errorbar(plot_years, plot_coefs, yerr=1.96*np.array(plot_ses),
                    fmt='o', color='#C0392B' if outcome == 'log_emp' else '#2980B9',
                    linewidth=2, capsize=5, markersize=6, label='beta +/- 1.96 SE')
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.axvline(2022, color='gray', linewidth=1.5, linestyle='--', label='ChatGPT Launch (ref.)')
        ax.axvspan(2019.5, 2021.5, color='gray', alpha=0.15, label='COVID years (included here)')
        ax.set_xlabel('Year')
        ax.set_ylabel(f'Coefficient on {label} x Year')
        ax.set_title(f'{label} Event Study: {outcome_titles[outcome]} (COVID-inclusive, no control)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(fontsize=9)
        plt.tight_layout()
        fname = f'event_study_{label.lower()}_{outcome}_covid_inclusive_nocontrol.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {fname}')


## Summary table: 4 rows (2 indices x 2 outcomes)


In [ ]:
rows = []
for (label, outcome), res in pooled_results.items():
    rows.append({
        'index': label,
        'outcome': outcome,
        'spec': 'COVID-inclusive, no control',
        'coef': res.params['did'],
        'se': res.std_errors['did'],
        'pvalue': res.pvalues['did'],
        'n_obs': res.nobs,
    })

summary = pd.DataFrame(rows).sort_values(['index', 'outcome']).reset_index(drop=True)
summary.to_csv('did_covid_inclusive_nocontrol_summary.csv', index=False)
print(summary.to_string(index=False))
print()
print(f'Total rows: {len(summary)} (expected 4 = 2 indices x 2 outcomes)')
